# KSPP pseudopotentials + QE input

Minimal QEpy workflow:

```python
qe_options = {...}
atoms = bulk("Al", "fcc", a=4.05, cubic=True)
pwin = QEInput(qe_options=qe_options, atoms=atoms, ksppresolver=True)
pw_driver = Driver(pwin)
```

More control:

```python
pwin = QEInput(qe_options=qe_options, atoms=atoms, ksppresolver=True)
pwin.ksppresolver(xc="LDA", offline=True, search_paths=[kspp_root])
pw_driver = Driver(pwin)
```

Different KSPP table (e.g. NC PseudoDojo):

```python
pwin.ksppresolver(table="norm-conserving/nc-sr-04", accuracy="stringent")
```

Sections below walk through the same ideas step by step.

In [1]:
from pathlib import Path

from ase.build import bulk
from qepy.io import QEInput

NOTEBOOK_DIR = Path(".").resolve()
PSEUDO_DIR = NOTEBOOK_DIR / "pseudos"

atoms = bulk("Al", "fcc", a=4.05, cubic=True)
qe_options = {
    "&control": {"calculation": "'scf'"},
    "&system": {
        "ibrav": 0,
        "degauss": 0.005,
        "ecutwfc": 30,
        "occupations": "'smearing'",
    },
    "&electrons": {"mixing_beta": 0.5},
    "k_points automatic": ["2 2 2 0 0 0"],
}

## Minimal: `QEInput(ksppresolver=True)` + `Driver(pwin)`

KSPP fills `pseudo_dir` and `atomic_species` automatically. Below we only preview the QE input (full `Driver` run needs QE installed).

In [2]:
pwin = QEInput(qe_options=qe_options, atoms=atoms, ksppresolver=True)

print("pseudo_dir:", pwin.qe_options["&control"]["pseudo_dir"])
print("atomic_species:", pwin.qe_options["atomic_species"])

input_path = NOTEBOOK_DIR / "al_minimal.in"
pwin.write_qe_input(input_path, atoms=atoms, qe_options=pwin.qe_options)
print(input_path.read_text())

# pw_driver = Driver(pwin)

pseudo_dir: '/Users/michele/.cache/qepy/kspp/ultrasoft/gbrv-v1.5/PBE/standard/upf/'
atomic_species: ['Al    26.981538 al_pbe_v1.uspp.F.UPF']
&CONTROL
   calculation = 'scf'
   pseudo_dir = '/Users/michele/.cache/qepy/kspp/ultrasoft/gbrv-v1.5/PBE/standard/upf/'
/

&SYSTEM
   ibrav = 0
   degauss = 0.005
   ecutwfc = 30
   occupations = 'smearing'
   ntyp = 1
   nat = 4
/

&ELECTRONS
   mixing_beta = 0.5
/

&IONS
/

&CELL
/

&FCP
/

&RISM
/

ATOMIC_SPECIES
Al    26.981538 al_pbe_v1.uspp.F.UPF

K_POINTS automatic
2 2 2 0 0 0

CELL_PARAMETERS angstrom
4.05000000000000 0.00000000000000 0.00000000000000
0.00000000000000 4.05000000000000 0.00000000000000
0.00000000000000 0.00000000000000 4.05000000000000

ATOMIC_POSITIONS angstrom
Al   0.00000000000000 0.00000000000000 0.00000000000000
Al   0.00000000000000 2.02500000000000 2.02500000000000
Al   2.02500000000000 0.00000000000000 2.02500000000000
Al   2.02500000000000 2.02500000000000 0.00000000000000




## Advanced: configure with `pwin.ksppresolver(...)`

Change table, XC, offline mode, or manual overrides before passing to `Driver`.

In [3]:
KSPP_ROOT = (NOTEBOOK_DIR.parents[2].parent.parent / "KSPP").resolve()

pwin = QEInput(qe_options=dict(qe_options), atoms=atoms, ksppresolver=True)
pwin.ksppresolver(
    xc="LDA",
    search_paths=[KSPP_ROOT] if KSPP_ROOT.is_dir() else None,
    offline=KSPP_ROOT.is_dir(),
)

print("atomic_species:", pwin.qe_options["atomic_species"])

input_path = NOTEBOOK_DIR / "al_lda.in"
pwin.write_qe_input(input_path, atoms=atoms, qe_options=pwin.qe_options)
print(input_path.read_text())

atomic_species: ['Al    26.981538 al_lda_v1.uspp.F.UPF']
&CONTROL
   calculation = 'scf'
   pseudo_dir = '/Users/michele/.cache/qepy/kspp/ultrasoft/gbrv-v1.5/LDA/standard/upf/'
/

&SYSTEM
   ibrav = 0
   degauss = 0.005
   ecutwfc = 30
   occupations = 'smearing'
   ntyp = 1
   nat = 4
/

&ELECTRONS
   mixing_beta = 0.5
/

&IONS
/

&CELL
/

&FCP
/

&RISM
/

ATOMIC_SPECIES
Al    26.981538 al_lda_v1.uspp.F.UPF

K_POINTS automatic
2 2 2 0 0 0

CELL_PARAMETERS angstrom
4.05000000000000 0.00000000000000 0.00000000000000
0.00000000000000 4.05000000000000 0.00000000000000
0.00000000000000 0.00000000000000 4.05000000000000

ATOMIC_POSITIONS angstrom
Al   0.00000000000000 0.00000000000000 0.00000000000000
Al   0.00000000000000 2.02500000000000 2.02500000000000
Al   2.02500000000000 0.00000000000000 2.02500000000000
Al   2.02500000000000 2.02500000000000 0.00000000000000




## NC table: `norm-conserving/nc-sr-04` (stringent)

Switch to a **norm-conserving** PseudoDojo table instead of the default GBRV ultrasoft library.

NC tables in KSPP use accuracy levels `standard` (softer) and `stringent` (harder, higher accuracy). Filenames are element-based, e.g. `Al.upf`.

In [4]:
KSPP_ROOT = (NOTEBOOK_DIR.parents[2].parent.parent / "KSPP").resolve()

pwin = QEInput(qe_options=dict(qe_options), atoms=atoms, ksppresolver=True)
pwin.ksppresolver(
    table="norm-conserving/nc-sr-04",
    xc="PBE",
    accuracy="stringent",
    fmt="upf",
#    search_paths=[KSPP_ROOT] if KSPP_ROOT.is_dir() else None,
#    offline=KSPP_ROOT.is_dir(),
)

print("pseudo_dir:", pwin.qe_options["&control"]["pseudo_dir"])
print("atomic_species:", pwin.qe_options["atomic_species"])

input_path = NOTEBOOK_DIR / "al_nc_stringent.in"
pwin.write_qe_input(input_path, atoms=atoms, qe_options=pwin.qe_options)
print(input_path.read_text())

pseudo_dir: '/Users/michele/.cache/qepy/kspp/norm-conserving/nc-sr-04/PBE/stringent/upf/'
atomic_species: ['Al    26.981538 Al.upf']
&CONTROL
   calculation = 'scf'
   pseudo_dir = '/Users/michele/.cache/qepy/kspp/norm-conserving/nc-sr-04/PBE/stringent/upf/'
/

&SYSTEM
   ibrav = 0
   degauss = 0.005
   ecutwfc = 30
   occupations = 'smearing'
   ntyp = 1
   nat = 4
/

&ELECTRONS
   mixing_beta = 0.5
/

&IONS
/

&CELL
/

&FCP
/

&RISM
/

ATOMIC_SPECIES
Al    26.981538 Al.upf

K_POINTS automatic
2 2 2 0 0 0

CELL_PARAMETERS angstrom
4.05000000000000 0.00000000000000 0.00000000000000
0.00000000000000 4.05000000000000 0.00000000000000
0.00000000000000 0.00000000000000 4.05000000000000

ATOMIC_POSITIONS angstrom
Al   0.00000000000000 0.00000000000000 0.00000000000000
Al   0.00000000000000 2.02500000000000 2.02500000000000
Al   2.02500000000000 0.00000000000000 2.02500000000000
Al   2.02500000000000 2.02500000000000 0.00000000000000




/Users/michele/Documents/hackathon/test_qepy/env/lib/python3.10/site-packages/qepy/io.py:348: UserWarning: KSPP: user ecutwfc=30.0 Ry is below the pseudopotential suggestion (37.5 Ry).
  self.apply_kspp(


### Ecut warnings from the UPF

KSPP reads suggested cutoffs from the resolved UPF files. By default QEpy **warns** if your `ecutwfc` / `ecutrho` are below the suggestion. Pass `update_ecuts=True` to bump them automatically.

In [5]:
import warnings

KSPP_ROOT = (NOTEBOOK_DIR.parents[2].parent.parent / "KSPP").resolve()

pwin = QEInput(qe_options=dict(qe_options), atoms=atoms, ksppresolver=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    pwin.ksppresolver(
        table="norm-conserving/nc-sr-04",
        accuracy="stringent",
        search_paths=[KSPP_ROOT] if KSPP_ROOT.is_dir() else None,
        offline=KSPP_ROOT.is_dir(),
    )
    for item in caught:
        print(item.message)

# Auto-update cutoffs to the PP suggestion
pwin = QEInput(qe_options=dict(qe_options), atoms=atoms, ksppresolver=True)
pwin.ksppresolver(
    table="norm-conserving/nc-sr-04",
    accuracy="stringent",
    search_paths=[KSPP_ROOT] if KSPP_ROOT.is_dir() else None,
    offline=KSPP_ROOT.is_dir(),
    update_ecuts=True,
)
print("ecutwfc:", pwin.qe_options["&system"]["ecutwfc"])
print("ecutrho:", pwin.qe_options["&system"]["ecutrho"])

KSPP: user ecutwfc=30.0 Ry is below the pseudopotential suggestion (37.5 Ry).
ecutwfc: 37.5
ecutrho: 150.0


/Users/michele/Documents/hackathon/test_qepy/env/lib/python3.10/site-packages/qepy/io.py:348: UserWarning: KSPP: user ecutwfc=30.0 Ry is below the pseudopotential suggestion (37.5 Ry).
  self.apply_kspp(


## On disk: explicit `atomic_species`

Classic workflow without KSPP auto-resolution.

In [6]:
pwin = QEInput()
qe_disk = dict(qe_options)

PSEUDO_DIR.mkdir(exist_ok=True)
upf_name = "al_pbe_v1.uspp.F.UPF"
upf_path = PSEUDO_DIR / upf_name
if not upf_path.is_file():
    upf_path.write_bytes(QEInput.KSPPResolver().resolve("Al").read_bytes())

qe_disk["&control"]["pseudo_dir"] = f"'{PSEUDO_DIR}/'"
qe_disk["atomic_species"] = [f"Al  26.981538 {upf_name}"]

input_path = NOTEBOOK_DIR / "al_disk.in"
pwin.write_qe_input(input_path, atoms=atoms, qe_options=qe_disk)
print(input_path.read_text())

&CONTROL
   calculation = 'scf'
   pseudo_dir = '/Users/michele/Documents/hackathon/QEPy_on_mac/QEpy/examples/jupyter/scf/pseudos/'
/

&SYSTEM
   ibrav = 0
   degauss = 0.005
   ecutwfc = 37.5
   occupations = 'smearing'
   ntyp = 1
   nat = 4
   ecutrho = 150.0
/

&ELECTRONS
   mixing_beta = 0.5
/

&IONS
/

&CELL
/

&FCP
/

&RISM
/

ATOMIC_SPECIES
Al  26.981538 al_pbe_v1.uspp.F.UPF

K_POINTS automatic
2 2 2 0 0 0

CELL_PARAMETERS angstrom
4.05000000000000 0.00000000000000 0.00000000000000
0.00000000000000 4.05000000000000 0.00000000000000
0.00000000000000 0.00000000000000 4.05000000000000

ATOMIC_POSITIONS angstrom
Al   0.00000000000000 0.00000000000000 0.00000000000000
Al   0.00000000000000 2.02500000000000 2.02500000000000
Al   2.02500000000000 0.00000000000000 2.02500000000000
Al   2.02500000000000 2.02500000000000 0.00000000000000


